In [ ]:
# CREATE PROPERTY PREDICTION HEATMAP
import types
import ipywidgets as widgets
import pandas as pd, numpy as np
import pathlib

import stages.utils.pdaa as pdaa
import stages.utils.pubchem as pubchem
import stages.utils.sparql as sparql
import stages.utils.openai as openai

import rdflib

from rdflib import RDF
from IPython.display import display, HTML

from tqdm import tqdm
from matplotlib import pyplot as plt
import seaborn as sns

from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, leaves_list
import plotly.graph_objects as go
from tqdm import tqdm

# setup ===============================
cachedir = pathlib.Path('cache') / 'notebooks' / 'pdaa.ipynb'
cachedir.mkdir(parents=True, exist_ok=True)

# TODO somehow multiple uris map to the same title. This needs to be fixed in chemharmony.
# for now we add a number to differentiate duplicates
DCTERMS = rdflib.Namespace('http://purl.org/dc/elements/1.1/')
def get_uri_titles():
    # Get titles for URIs from graph
    uri_titles = sparql.Query(pdaa.pdaa_graph, cachedir / 'pdaa_graph') \
        .select_typed({'uri':str,'title':str}) \
        .where(f'?ppuri <{DCTERMS.term("title")}> ?title') \
        .where(f'?ppuri <{RDF.type}> toxindex:predicted_property') \
        .where(f'?ppuri <{DCTERMS.term("has_identifier")}> ?uri') \
        .cache_execute().groupby('uri').first().reset_index()

    # Add numbers to make duplicate titles unique
    for title in uri_titles['title'].value_counts()[uri_titles['title'].value_counts() > 1].index:
        mask = uri_titles['title'] == title
        uri_titles.loc[mask, 'title'] = [f"{title}_{i+1}" for i in range(sum(mask))]
        
    return uri_titles
uri_titles = get_uri_titles()

def parse_chemical_list(chemical_list) -> dict[str, str]:
    chemicals = {}
    for line in filter(None, map(str.strip, chemical_list.split('\n'))):
        alias, name = line.split(':', 1) if ':' in line else (line, line)
        chemicals[alias.strip()] = name.strip()
    return chemicals


# widgets ===============================
# predictions should be a dataframe with columns ['uri','title','chemical_name','prediction']
def build_heatmap(predictions, midvalue=0.5, max_val=None):
    # Pivot the predictions into a matrix
    pdf = predictions.pivot(index='title', columns='chemical_name', values='prediction')

    # Perform hierarchical clustering on rows and columns
    row_linkage = linkage(pdf.values, method='ward', metric='euclidean')
    col_linkage = linkage(pdf.T.values, method='ward', metric='euclidean')

    # Get the order of rows and columns based on clustering
    row_order = leaves_list(row_linkage)
    col_order = leaves_list(col_linkage)

    # Reorder the DataFrame
    pdf_clustered = pdf.iloc[row_order, col_order]

    # Create clickable row labels using the 'uri' column
    row_links = dict(zip(predictions['title'], predictions['uri']))
    row_labels = [f'<a href="{row_links[title]}" target="_blank">{title}</a>' if title in row_links else title for title in pdf_clustered.index]

    # Get min and max values for colorscale
    min_val = pdf_clustered.values.min()
    max_val = pdf_clustered.values.max() if max_val is None else max_val

    # rescale z so that if it is bigger than max_val, it is set to max_val and if it is smaller than min_val, it is set to min_val
    pdf_clustered_clipped = pdf_clustered.clip(min_val, max_val)

    # Create blue gradient up to midvalue
    colorscale = [[0, 'rgb(0,0,255)'],[1, 'rgb(255,0,0)']]
    # Define the heatmap trace with dynamic min/max values
    heatmap = go.Heatmap(
        z=pdf_clustered_clipped.values,
        x=pdf_clustered_clipped.columns,
        y=row_labels,
        colorscale=colorscale,  # Blue to black to red
        zmin=min_val,
        zmax=max_val,
        colorbar=dict(title="", orientation="v", x=-0.2, y=0.5, thickness=20),
        xgap=2,  # Increased gap between cells to make boxes appear smaller
        ygap=2,  # Increased gap between cells to make boxes appear smaller
        showscale=True,
        text=[[f'{val:.2f}' for val in row] for row in pdf_clustered.values],  # Show values in cells
        texttemplate='%{text}',
        textfont={"color": "white"}  # White text
    )

    # Create the layout
    layout = go.Layout(
        title=f'Property Predictions Heatmap',
        xaxis=dict(
            title="Chemical Names",
            gridcolor='black',  # Make x-axis grid black
            showgrid=False
        ),
        yaxis=dict(
            title="Titles", 
            tickmode="array", 
            tickvals=list(range(len(pdf.index))), 
            ticktext=row_labels,
            side='right',  # Move labels to right side
            gridcolor='black',  # Make y-axis grid black
            showgrid=False,
            tickfont=dict(size=12),  # Adjust font size if needed
            tickprefix='   ',  # Add padding after labels
            ticksuffix='     '  # Add padding before labels
        ),
        height=800,
        plot_bgcolor='black',  # Make gaps appear black
        margin=dict(r=350)  # Increased right margin for more space
    )

    # Generate the figure
    fig = go.Figure(data=[heatmap], layout=layout)
    
    fig.show()

all_ao = sparql.Query(pdaa.pdaa_graph, cachedir / 'pdaa_graph') \
    .select_typed({'aop':str, 'mie':str, 'ao':str, 'ao_title':str, 'mie_title':str}) \
    .where(f'?aop <{RDF.type}> aop:AdverseOutcomePathway') \
    .where(f'?aop aop:has_adverse_outcome ?ao') \
    .where(f'?aop aop:has_molecular_initiating_event ?mie') \
    .where(f'?ao <{DCTERMS.term("title")}> ?ao_title') \
    .where(f'?mie <{DCTERMS.term("title")}> ?mie_title') \
    .cache_execute()

all_pp = sparql.Query(pdaa.pdaa_graph, cachedir / 'pdaa_graph') \
    .select_typed({'pp':str, 'pp_title':str}) \
    .where(f'?pp <{RDF.type}> toxindex:predicted_property') \
    .where(f'?pp <{DCTERMS.term("title")}> ?pp_title') \
    .cache_execute()
# LINK MIE TO PROPERTY TOKENS ===============================================
predicted_property_identifiers = pdaa.proptoken_uris['uri'].unique()
mies = all_ao['mie'].unique()
mie_proptoken_id_simtable = pdaa.get_uri_similars(mies, predicted_property_identifiers).query('similarity > 0.5')
mie_proptoken_id_simtable.columns = ['mie', 'property_token_id_uri', 'similarity']
mie_proptoken_id_simtable = mie_proptoken_id_simtable.sort_values('similarity', ascending=False).drop_duplicates('property_token_id_uri', keep='first')

def get_all_predictions(inchi_list):
    pb = tqdm(total=len(inchi_list), desc="Predicting properties")
    all_predictions = pdaa.predict_all_properties_with_sqlite_cache(inchi_list, pb)
    preds_df = pd.DataFrame(all_predictions, columns=['inchi', 'token', 'prediction'])
    preds_df = preds_df.merge(pdaa.proptoken_uris[['uri','token']], on='token')[['uri','inchi','prediction']]
    preds_df.rename(columns={'uri':'property_token_id_uri'}, inplace=True)
    
    all_pred_df = preds_df.merge(mie_proptoken_id_simtable, on='property_token_id_uri')
    all_pred_df['weight'] = all_pred_df['similarity'] * all_pred_df['prediction']
    return all_pred_df[['mie','property_token_id_uri','inchi','similarity','prediction','weight']]


def get_property_predictions(predictions, promptstr):
    # get relevant property predictions
    prompt_property = pdaa.get_prompt_similars(promptstr, pdaa.proptoken_uris['uri'].unique(), top_k_to_search=10000)
    prompt_property = prompt_property.query('similarity > 0.4')
    
    res = predictions[['property_token_id_uri','chemical_name','prediction']]
    res = res[res['property_token_id_uri'].isin(prompt_property['uri'])]
    res = res.merge(uri_titles, left_on='property_token_id_uri', right_on='uri')
    return res[['uri','title','chemical_name','prediction']]

def build_property_heatmap(output_widget, predictions):
    with output_widget:
        output_widget.clear_output()
        build_heatmap(predictions, midvalue=0.5, max_val=1)

def get_mie_predictions(predictions, prompt):
    # get relevant mie predictions
    prompt_mie = pdaa.get_prompt_similars(prompt, all_ao['mie'].unique(), top_k_to_search=10000)
    prompt_mie = prompt_mie.query('similarity > 0.4')
    prompt_mie = all_ao[all_ao['mie'].isin(prompt_mie['uri'])][['mie','mie_title']]

    mie_predictions = predictions.merge(prompt_mie, on=['mie'])
    mie_df = mie_predictions.groupby(['mie','mie_title','chemical_name'])['prediction'].max().reset_index()
    
    # need 'uri','title','chemical_name','prediction'
    mie_weights = mie_df[['mie','mie_title','chemical_name','prediction']]
    mie_weights.rename(columns={'mie':'uri','mie_title':'title'}, inplace=True)

    top_mie = mie_weights.groupby('uri')['prediction'].sum().sort_values(ascending=False).index[:30]
    mie_weights = mie_weights[mie_weights['uri'].isin(top_mie)]
    mie_weights['title'] = mie_weights['title'].str[:40]

    return mie_weights[['uri','title','chemical_name','prediction']]

def build_mie_heatmap(output_widget, mie_predictions):
    with output_widget:
        output_widget.clear_output()
        if len(mie_predictions['uri'].unique()) < 3:
            raise ValueError(f"Prompt {prompt} did not return relevant key events. Try a different prompt.")
        build_heatmap(mie_predictions, midvalue=0.5, max_val=1)

def build_mie_barchart(widget, predictions):
    with widget:
        widget.clear_output()
        plt.figure(figsize=(15, 6))
        ax = predictions.groupby('chemical_name')['prediction'].sum().sort_values(ascending=False).plot.bar(
            color='#485B8F',
            edgecolor='#91403C',
            linewidth=1
        )
        plt.title(f'Total Relevant MIE Predictions by Chemical')
        plt.xlabel('Chemical')
        plt.ylabel('Sum of Predictions')
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.show()

# WIDGETS ===============================================
heatmap_type = widgets.ToggleButtons(
    options=['Property Heatmap', 'MIE Heatmap'],
    description='Display:',
    style={'description_width': 'initial'},
    button_style='primary',  # Changed from 'info' to 'primary' for better contrast
    value='Property Heatmap'  # Set default selected value
)
heatmap_type.layout.margin = '10px 0'

alias_default = """DEHP:bis(2-ethylhexyl) phthalate
DIUP:diisoundecyl phthalate
DTDP:ditridecyl phthalate
DIDP:diisodecyl phthalate
DINP:diisononyl phthalate
sucrose"""
chemical_list = widgets.Textarea(
    value=alias_default,
    description='enter `alias:name` pairs',
    layout=widgets.Layout(width='800px', height="auto", overflow="visible")
)
prompt = widgets.Text(value='endocrine disruption', description='Prompt:')
go_btn = widgets.Button(description='go')
heatmap_output = widgets.Output(layout=widgets.Layout(height="auto", overflow="visible"))
barchart_output = widgets.Output(layout=widgets.Layout(height="auto", overflow="visible"))

def build_example_buttons():
    example_1 = widgets.Button(
        description='Example 1: Endocrine Disruption',
        button_style='info',
        layout=widgets.Layout(width='250px')
    )

    example_2 = widgets.Button(
        description='Example 2: Liver Toxicity', 
        button_style='info',
        layout=widgets.Layout(width='250px')
    )

    example_3 = widgets.Button(
        description='Example 3: Developmental Effects',
        button_style='info', 
        layout=widgets.Layout(width='250px')
    )

    def update_example_1(b):
        chemical_list.value = """DEHP:bis(2-ethylhexyl) phthalate
    DINP:diisononyl phthalate 
    DIDP:diisodecyl phthalate
    BPA:bisphenol A
    TBBPA:tetrabromobisphenol A"""
        prompt.value = "endocrine disruption thyroid hormone"
        update_charts(None)

    def update_example_2(b):
        chemical_list.value = """acetaminophen:acetaminophen
    ethanol:ethanol
    CCl4:carbon tetrachloride
    aspirin:acetylsalicylic acid
    APAP:acetaminophen"""
        prompt.value = "liver damage hepatotoxicity"
        update_charts(None)

    def update_example_3(b):
        chemical_list.value = """thalidomide:thalidomide
    valproic_acid:valproic acid
    isotretinoin:isotretinoin
    warfarin:warfarin
    ethanol:ethanol"""
        prompt.value = "birth defects teratogenicity"
        update_charts(None)

    example_1.on_click(update_example_1)
    example_2.on_click(update_example_2) 
    example_3.on_click(update_example_3)
    return example_1, example_2, example_3

example_1, example_2, example_3 = build_example_buttons()
examples_box = widgets.HBox([example_1, example_2, example_3])
status_label = widgets.Label(value='Status: Ready')

def update_charts(change):
    chemicals = parse_chemical_list(chemical_list.value)
    chemical_inchi = [pubchem.lookup_chemical_inchi(chemical_name) for chemical_name in chemicals.values()]
    inchi2key = dict(zip(chemical_inchi, chemicals.keys()))
    
    status_label.value = 'Status: Getting predictions...'
    predictions = get_all_predictions(chemical_inchi)
    predictions['chemical_name'] = predictions['inchi'].map(lambda x: inchi2key[x])

    if heatmap_type.value == 'Property Heatmap':
        status_label.value = 'Status: Getting property predictions...'
        property_predictions = get_property_predictions(predictions, prompt.value)
        status_label.value = 'Status: Building property heatmap...'
        build_property_heatmap(heatmap_output, property_predictions)
        barchart_output.clear_output()
    else:
        status_label.value = 'Status: Getting MIE predictions...'
        mie_predictions = get_mie_predictions(predictions, prompt.value)
        status_label.value = 'Status: Building MIE heatmap...'
        build_mie_heatmap(heatmap_output, mie_predictions)
        status_label.value = 'Status: Building MIE barchart...'
        build_mie_barchart(barchart_output, mie_predictions)
    status_label.value = 'Status: Ready'

# Link the button to the update function
heatmap_type.observe(update_charts, names='value')

# display widgets ===============================
display(widgets.HTML("<h3>Find & Rank Adverse Outcomes for a Chemical & Prompt</h3>"))
display(examples_box)
display(status_label)
display(chemical_list, prompt, heatmap_type, heatmap_output, barchart_output)
update_charts(None)
